# 02 — Moderation analysis

Fits the moderated OLS model and reports simple slopes and Johnson–Neyman bounds.

**Requires:** `data/adaptive_reserve_moderation_subject_level.csv` from `01_data.ipynb`.


In [1]:
# Adaptive Reserve — Moderation Analysis
#
# Loads subject-level data produced by `01_data.ipynb` and
# fits the moderated regression reported in the manuscript.

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from statsmodels.api import OLS, add_constant

warnings.filterwarnings("ignore")


def _package_dir(cwd=None):
    root = (cwd or Path.cwd()).resolve()
    if (root / "data").is_dir() and (root / "notebooks").is_dir():
        return root
    if root.name == "notebooks" and (root.parent / "data").is_dir():
        return root.parent
    if (root / "01_analysis.ipynb").exists() or (root / "02_plot.ipynb").exists():
        return root
    pkg = root / "adaptive_reserve_moderation_analysis"
    if (pkg / "notebooks" / "01_analysis.ipynb").exists():
        return pkg
    return root


PACKAGE_DIR = _package_dir()
DATA_CSV = PACKAGE_DIR / "data" / "adaptive_reserve_moderation_subject_level.csv"

X_COL = "global_modularity_auc"
Y_COL = "rt_transition_diff__flexibility_stability"
M_COL = "rt_duration_diff__long-short"


def zscore(s):
    return (s - s.mean()) / s.std(ddof=1)


def fit_moderation(df):
    x = zscore(df["global_modularity_auc"])
    m = zscore(df["rt_duration_diff__long-short"])
    y = zscore(df["rt_transition_diff__flexibility_stability"])
    xm = x * m
    design = add_constant(np.column_stack([x, m, xm]))
    model = OLS(y, design).fit()
    cov = np.asarray(model.cov_params())
    b1 = float(model.params.iloc[1])
    b3 = float(model.params.iloc[3])
    v11, v13, v33 = cov[1, 1], cov[1, 3], cov[3, 3]
    df_resid = model.df_resid
    return {
        "model": model,
        "x_z": x,
        "m_z": m,
        "y_z": y,
        "b1": b1,
        "b3": b3,
        "v11": v11,
        "v13": v13,
        "v33": v33,
        "df_resid": df_resid,
        "df_raw": df,
    }


def simple_slope(b1, b3, m, v11, v13, v33, df):
    theta = b1 + b3 * m
    var = v11 + 2 * m * v13 + m * m * v33
    se = np.sqrt(max(var, 0))
    t = theta / se if se > 0 else np.nan
    p = 2 * (1 - scipy_stats.t.cdf(abs(t), df)) if np.isfinite(t) else np.nan
    tc = scipy_stats.t.ppf(0.975, df)
    return theta, se, t, p, theta - tc * se, theta + tc * se


def _jn_roots(b1, b3, v11, v13, v33, df):
    tc = scipy_stats.t.ppf(0.975, df)
    t2 = tc * tc
    A = b3 * b3 - t2 * v33
    B = 2 * (b1 * b3 - t2 * v13)
    C = b1 * b1 - t2 * v11
    if abs(A) < 1e-12:
        if abs(B) < 1e-12:
            return []
        return [-C / B]
    disc = B * B - 4 * A * C
    if disc < 0:
        return []
    s = np.sqrt(disc)
    return sorted([(-B - s) / (2 * A), (-B + s) / (2 * A)])


def slope_curve(fit, m_grid):
    b1, b3 = fit["b1"], fit["b3"]
    v11, v13, v33, df = fit["v11"], fit["v13"], fit["v33"], fit["df_resid"]
    theta = b1 + b3 * m_grid
    var = v11 + 2 * m_grid * v13 + m_grid ** 2 * v33
    se = np.sqrt(np.maximum(var, 0))
    tc = scipy_stats.t.ppf(0.975, df)
    return theta, theta - tc * se, theta + tc * se


def predicted_y_z(x_z, m_z, fit):
    model = fit["model"]
    b0, b1, b2, b3 = [float(v) for v in model.params.iloc[:4]]
    return b0 + b1 * x_z + b2 * m_z + b3 * x_z * m_z


fig_df = pd.read_csv(DATA_CSV)
print(f"N = {len(fig_df)} subjects")

fig_fit = fit_moderation(fig_df)
_model = fig_fit["model"]
print(
    f"F({_model.df_model:.0f}, {_model.df_resid:.0f}) = {_model.fvalue:.2f}, "
    f"p = {_model.f_pvalue:.4f}, R2 = {_model.rsquared:.3f}"
)
print(
    f"b_X = {_model.params.iloc[1]:.3f}, p = {_model.pvalues.iloc[1]:.3f}; "
    f"b_M = {_model.params.iloc[2]:.3f}, p = {_model.pvalues.iloc[2]:.3f}; "
    f"b_int = {_model.params.iloc[3]:.3f}, p = {_model.pvalues.iloc[3]:.4f}"
)

jn_roots = _jn_roots(
    fig_fit["b1"], fig_fit["b3"],
    fig_fit["v11"], fig_fit["v13"], fig_fit["v33"], fig_fit["df_resid"],
)
print(f"Johnson-Neyman roots (M_Z): {jn_roots}")
for _lab, _mz in [("-1 SD", -1), ("Mean", 0), ("+1 SD", 1)]:
    _th, _se, _t, _p, _, _ = simple_slope(
        fig_fit["b1"], fig_fit["b3"], _mz,
        fig_fit["v11"], fig_fit["v13"], fig_fit["v33"], fig_fit["df_resid"],
    )
    print(f" {_lab}: b = {_th:.3f}, SE = {_se:.3f}, p = {_p:.4f}")


N = 43 subjects
F(3, 39) = 4.38, p = 0.0094, R2 = 0.252
b_X = -0.147, p = 0.311; b_M = -0.070, p = 0.645; b_int = 0.389, p = 0.0060
Johnson-Neyman roots (M_Z): [np.float64(-0.36048865077630654), np.float64(2.204632919323197)]
 -1 SD: b = -0.536, SE = 0.176, p = 0.0042
 Mean: b = -0.147, SE = 0.143, p = 0.3114
 +1 SD: b = 0.243, SE = 0.214, p = 0.2630
